<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/Nlp/word2vec_glove_fasttext_sifirdan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kelime Temsilleri Sıfırdan: TF-IDF → Word2Vec → GloVe → FastText

`embedding_karsilastirma.ipynb` defterinde bu yöntemleri hazır kütüphanelerle (gensim) karşılaştırmıştık. Burada her birini **PyTorch ile sıfırdan** yazıp içlerinde ne olduğunu göreceğiz.

| # | Yöntem | Temel fikir |
|---|---|---|
| 1 | One-hot / TF-IDF | Seyrek (sparse) vektörler, kelimeler arası benzerlik yok |
| 2 | Word2Vec (CBOW, Skip-gram) | Yerel pencerede komşu kelimeyi tahmin et + **Negative Sampling** |
| 3 | GloVe | **Küresel eş-oluşum matrisinin** logaritmasını vektör çarpımıyla yaklaşıkla |
| 4 | FastText | Kelime = **karakter n-gram** vektörlerinin toplamı → sözlük dışı kelimelere de vektör |

In [ ]:
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

random.seed(0); np.random.seed(0); torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 100
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", cihaz)

## Korpus

Anlamsal yapısını bildiğimiz sentetik bir korpus kullanıyoruz: 4 kategori (hayvan, meyve, şehir, araç), her kategori kendine özgü bağlamlarda geçiyor. Böylece öğrenilen vektörlerin **aynı kategorideki kelimeleri yakınlaştırıp yakınlaştırmadığını** net biçimde test edebiliriz.

> *"You shall know a word by the company it keeps."* — J. R. Firth (1957). Dağılımsal hipotez: benzer bağlamlarda geçen kelimeler benzer anlamlıdır.

In [ ]:
kategoriler = {
    "hayvan": ["kedi", "köpek", "kuş", "at", "inek", "tavşan"],
    "meyve":  ["elma", "armut", "muz", "çilek", "kiraz", "üzüm"],
    "şehir":  ["istanbul", "ankara", "izmir", "erzincan", "bursa", "trabzon"],
    "araç":   ["araba", "otobüs", "tren", "uçak", "gemi", "bisiklet"],
}
kalip_listesi = {
    "hayvan": ["{w} bahçede koşuyor", "{w} mama yiyor", "sevimli {w} uyuyor",
               "veteriner {w} muayene etti", "{w} su içiyor", "küçük {w} oynuyor"],
    "meyve":  ["ben {w} yedim", "{w} çok tatlı", "pazardan taze {w} aldım",
               "{w} suyu içtim", "olgun {w} topladık", "kilo {w} kaç lira"],
    "şehir":  ["{w} şehrine gittim", "{w} trafiği yoğun", "{w} belediyesi açıkladı",
               "yarın {w} yolculuğu var", "{w} merkezinde oturuyorum", "{w} havası soğuk"],
    "araç":   ["{w} ile yolculuk ettik", "{w} bileti aldım", "{w} hızla gidiyor",
               "{w} durağında bekledim", "{w} kalkmak üzere", "yeni {w} çok konforlu"],
}

cumleler = []
for _ in range(6000):
    kat = random.choice(list(kategoriler))
    cumleler.append(random.choice(kalip_listesi[kat]).format(w=random.choice(kategoriler[kat])).split())

kelime_kategorisi = {w: k for k, ws in kategoriler.items() for w in ws}
print("Örnek:", [" ".join(c) for c in cumleler[:5]])
print("Cümle sayısı:", len(cumleler), "| toplam token:", sum(map(len, cumleler)))

---
## 1) Seyrek temsiller: One-hot ve TF-IDF

**One-hot:** $|V|$ boyutlu, tek bir 1 içeren vektör. Her iki farklı kelimenin kosinüs benzerliği **0**'dır, `kedi` ile `köpek` arasındaki yakınlık temsil edilemez.

**TF-IDF:** Doküman vektörü; terim frekansını, terimin kaç dokümanda geçtiğine göre cezalandırır:

$$\text{tfidf}(t, d) = \text{tf}(t, d) \cdot \log\frac{1 + N}{1 + \text{df}(t)} + 1$$

(scikit-learn'ün "smooth" versiyonu). Her yerde geçen kelimeler düşük, ayırt edici kelimeler yüksek ağırlık alır.

In [ ]:
dokumanlar = [" ".join(c) for c in cumleler]
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(dokumanlar)

doluluk = X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])
print(f"TF-IDF matrisi: {X_tfidf.shape}  (doküman × sözlük)")
print(f"Sıfır olmayan oran: %{doluluk * 100:.2f}  → %{(1 - doluluk) * 100:.2f} seyrek")

idf = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).sort_values()
print("\nEn düşük IDF (her yerde geçen):", idf.head(5).round(2).to_dict())
print("En yüksek IDF (ayırt edici)    :", idf.tail(5).round(2).to_dict())

V_onehot = np.eye(3)
print("\nOne-hot kosinüs(kedi, köpek) =", float(V_onehot[0] @ V_onehot[1]), "→ anlamsal bilgi yok")

---
## 2) Word2Vec (Mikolov vd., 2013)

Her kelimeye öğrenilebilir, yoğun (dense) bir vektör atanır. İki mimari vardır:

| | **CBOW** (Continuous Bag of Words) | **Skip-gram** |
|---|---|---|
| Girdi → Çıktı | Bağlam kelimeleri → **merkez** kelime | Merkez kelime → **bağlam** kelimeleri |
| Örnek: `sevimli kedi uyuyor` | `[sevimli, uyuyor]` → `kedi` | `kedi` → `sevimli`, `kedi` → `uyuyor` |
| Hız | Daha hızlı | Daha yavaş |
| Güçlü yanı | Sık kelimeler | **Nadir kelimeler**, küçük veri |

### Negative Sampling

Tam softmax her adımda $|V|$ kelimenin hepsi üzerinden normalize eder, bu da büyük sözlükte çok pahalıdır. Negative Sampling problemi ikili sınıflandırmaya çevirir: *"Bu (merkez, bağlam) çifti gerçek mi, yoksa rastgele mi?"*

$$\mathcal{L} = -\log \sigma(\mathbf{u}_o^\top \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_k^\top \mathbf{v}_c)$$

- $\mathbf{v}_c$: merkez kelimenin **girdi** vektörü, $\mathbf{u}_o$: gerçek bağlam kelimesinin **çıktı** vektörü
- $\mathbf{u}_k$: $P(w) \propto \text{frekans}(w)^{0.75}$ dağılımından çekilen $K$ **negatif** kelime

In [ ]:
sayac = Counter(w for c in cumleler for w in c)
sozluk = ["<pad>"] + [w for w, _ in sayac.most_common()]
kelime2id = {w: i for i, w in enumerate(sozluk)}
V = len(sozluk)

frekans = torch.tensor([0] + [sayac[w] for w in sozluk[1:]], dtype=torch.float)
negatif_dagilim = frekans ** 0.75
negatif_dagilim /= negatif_dagilim.sum()

cumle_idleri = [[kelime2id[w] for w in c] for c in cumleler]
print("Sözlük boyutu:", V)

In [ ]:
PENCERE = 2

def skipgram_ciftleri(cumle_idleri, pencere):
    ciftler = []
    for c in cumle_idleri:
        for i, merkez in enumerate(c):
            for j in range(max(0, i - pencere), min(len(c), i + pencere + 1)):
                if j != i:
                    ciftler.append((merkez, c[j]))
    return torch.tensor(ciftler)

def cbow_ornekleri(cumle_idleri, pencere):
    baglamlar, merkezler = [], []
    for c in cumle_idleri:
        for i, merkez in enumerate(c):
            baglam = [c[j] for j in range(i - pencere, i + pencere + 1) if j != i and 0 <= j < len(c)]
            baglamlar.append(baglam + [0] * (2 * pencere - len(baglam)))   # <pad> ile doldur
            merkezler.append(merkez)
    return torch.tensor(baglamlar), torch.tensor(merkezler)

sg_ciftleri = skipgram_ciftleri(cumle_idleri, PENCERE)
cbow_baglam, cbow_merkez = cbow_ornekleri(cumle_idleri, PENCERE)
print("Skip-gram (merkez, bağlam) çifti:", len(sg_ciftleri))
print("CBOW örneği                     :", len(cbow_merkez))
print("İlk 5 skip-gram çifti:", [(sozluk[a], sozluk[b]) for a, b in sg_ciftleri[:5].tolist()])

In [ ]:
class SkipGramNS(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.giris = nn.Embedding(V, d)     # v: merkez kelime vektörleri (sonunda kullandığımız)
        self.cikis = nn.Embedding(V, d)     # u: bağlam kelime vektörleri
        nn.init.uniform_(self.giris.weight, -0.5 / d, 0.5 / d)
        nn.init.zeros_(self.cikis.weight)

    def forward(self, merkez, baglam, negatif):
        v = self.giris(merkez)                                    # (B, d)
        u_poz = self.cikis(baglam)                                # (B, d)
        u_neg = self.cikis(negatif)                               # (B, K, d)
        poz = F.logsigmoid((v * u_poz).sum(-1))                   # gerçek çift → σ(u·v) → 1
        neg = F.logsigmoid(-torch.bmm(u_neg, v.unsqueeze(-1)).squeeze(-1)).sum(-1)  # sahte → 0
        return -(poz + neg).mean()


class CBOWNS(SkipGramNS):
    def forward(self, baglam, merkez, negatif):
        maske = (baglam != 0).float().unsqueeze(-1)                       # <pad> pozisyonlarını yok say
        v = (self.giris(baglam) * maske).sum(1) / maske.sum(1).clamp(min=1)   # bağlam ortalaması
        u_poz = self.cikis(merkez)
        u_neg = self.cikis(negatif)
        poz = F.logsigmoid((v * u_poz).sum(-1))
        neg = F.logsigmoid(-torch.bmm(u_neg, v.unsqueeze(-1)).squeeze(-1)).sum(-1)
        return -(poz + neg).mean()


def egit(model, girdiler, hedefler, epok=5, yigin=512, K=5, lr=0.01):
    model.to(cihaz)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    kayiplar = []
    n = len(hedefler)
    for e in range(epok):
        perm = torch.randperm(n)
        toplam = 0.0
        for i in range(0, n, yigin):
            idx = perm[i:i + yigin]
            g, h = girdiler[idx].to(cihaz), hedefler[idx].to(cihaz)
            neg = torch.multinomial(negatif_dagilim, len(idx) * K, replacement=True).view(len(idx), K).to(cihaz)
            kayip = model(g, h, neg)
            opt.zero_grad(); kayip.backward(); opt.step()
            toplam += kayip.item() * len(idx)
        kayiplar.append(toplam / n)
        print(f"  epok {e + 1}: kayıp = {kayiplar[-1]:.4f}")
    return kayiplar

D = 50
print("Skip-gram eğitiliyor...")
skipgram = SkipGramNS(V, D)
sg_kayip = egit(skipgram, sg_ciftleri[:, 0], sg_ciftleri[:, 1])

print("\nCBOW eğitiliyor...")
cbow = CBOWNS(V, D)
cbow_kayip = egit(cbow, cbow_baglam, cbow_merkez)

### En yakın komşular ve 2B görselleştirme

In [ ]:
def normalize(m):
    return m / m.norm(dim=1, keepdim=True).clamp(min=1e-8)

def en_yakinlar(vektorler, kelime, n=5):
    E = normalize(vektorler)
    benzerlik = E @ E[kelime2id[kelime]]
    sirali = benzerlik.argsort(descending=True)
    return [(sozluk[i], round(benzerlik[i].item(), 2)) for i in sirali[1:n + 1].tolist()]

def komsu_tablosu(vektorler, kelimeler=("kedi", "elma", "ankara", "tren")):
    return pd.DataFrame({k: [f"{w} ({s})" for w, s in en_yakinlar(vektorler, k)] for k in kelimeler})

def pca_ciz(vektorler, baslik, ax):
    hedef = [w for ws in kategoriler.values() for w in ws]
    nokta = PCA(2).fit_transform(normalize(vektorler)[[kelime2id[w] for w in hedef]].cpu().numpy())
    renk = {"hayvan": "tab:orange", "meyve": "tab:red", "şehir": "tab:blue", "araç": "tab:green"}
    for (x, y), w in zip(nokta, hedef):
        ax.scatter(x, y, color=renk[kelime_kategorisi[w]], s=40)
        ax.annotate(w, (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.set_title(baslik)

sg_vektorleri = skipgram.giris.weight.detach().cpu()
cbow_vektorleri = cbow.giris.weight.detach().cpu()

print("Skip-gram en yakın komşular:")
display(komsu_tablosu(sg_vektorleri))
print("CBOW en yakın komşular:")
display(komsu_tablosu(cbow_vektorleri))

fig, eksenler = plt.subplots(1, 2, figsize=(15, 6))
pca_ciz(sg_vektorleri, "Skip-gram + Negative Sampling (PCA)", eksenler[0])
pca_ciz(cbow_vektorleri, "CBOW + Negative Sampling (PCA)", eksenler[1])
plt.tight_layout(); plt.show()

---
## 3) GloVe — Global Vectors (Pennington vd., 2014)

Word2Vec veriyi **pencere pencere** gezer. GloVe ise önce tüm korpus için bir **eş-oluşum matrisi** $X$ oluşturur ($X_{ij}$: $j$ kelimesinin $i$ kelimesinin bağlamında kaç kez geçtiği) ve sonra şu ağırlıklı en küçük kareler problemini çözer:

$$J = \sum_{i,j:\,X_{ij}>0} f(X_{ij})\left(\mathbf{w}_i^\top \tilde{\mathbf{w}}_j + b_i + \tilde b_j - \log X_{ij}\right)^2,
\qquad f(x) = \begin{cases}(x/x_{\max})^{\alpha} & x < x_{\max}\\ 1 & \text{aksi halde}\end{cases}$$

- $f$, çok sık çiftlerin (ör. "bir", "ve") kaybı domine etmesini engeller ($\alpha = 0.75$, $x_{\max} = 100$).
- Nihai vektör genellikle $\mathbf{w} + \tilde{\mathbf{w}}$ olarak alınır.

### Adım 1: Eş-oluşum matrisi (mesafeyle ağırlıklandırılmış)

In [ ]:
PENCERE_GLOVE = 3
X_es = torch.zeros(V, V)
for c in cumle_idleri:
    for i, wi in enumerate(c):
        for j in range(max(0, i - PENCERE_GLOVE), min(len(c), i + PENCERE_GLOVE + 1)):
            if j != i:
                X_es[wi, c[j]] += 1.0 / abs(i - j)     # yakın kelimeler daha fazla sayılır

ornek = ["kedi", "köpek", "elma", "ankara", "mama", "yedim", "şehrine", "bileti"]
idx = [kelime2id[w] for w in ornek]
pd.DataFrame(X_es[idx][:, idx].numpy().round(1), index=ornek, columns=ornek)

### GloVe'un sezgisi: olasılık oranları

$P(k \mid i) = X_{ik} / \sum_j X_{ij}$. Bir bağlam kelimesi $k$'nin iki kelimeyi ne kadar ayırt ettiği, oran $P(k|i)/P(k|j)$ ile ölçülür:
- $k$ yalnızca $i$ ile ilgiliyse oran **≫ 1**
- $k$ yalnızca $j$ ile ilgiliyse oran **≪ 1**
- $k$ ikisiyle de ilgili/ilgisizse oran **≈ 1**

In [ ]:
P = X_es / X_es.sum(1, keepdim=True).clamp(min=1e-8)
satirlar = []
for k in ["mama", "yedim", "taze", "uyuyor", "şehrine"]:
    p_kedi = P[kelime2id["kedi"], kelime2id[k]].item()
    p_elma = P[kelime2id["elma"], kelime2id[k]].item()
    satirlar.append({"bağlam k": k, "P(k|kedi)": p_kedi, "P(k|elma)": p_elma,
                     "oran": (p_kedi + 1e-6) / (p_elma + 1e-6)})
pd.DataFrame(satirlar).round(4)

### Adım 2: GloVe modelini eğitmek

In [ ]:
class GloVe(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.w = nn.Embedding(V, d)
        self.w_tilde = nn.Embedding(V, d)
        self.b = nn.Embedding(V, 1)
        self.b_tilde = nn.Embedding(V, 1)
        for e in [self.w, self.w_tilde]:
            nn.init.uniform_(e.weight, -0.5 / d, 0.5 / d)
        nn.init.zeros_(self.b.weight); nn.init.zeros_(self.b_tilde.weight)

    def forward(self, i, j, x, x_max=100.0, alfa=0.75):
        tahmin = (self.w(i) * self.w_tilde(j)).sum(-1) + self.b(i).squeeze(-1) + self.b_tilde(j).squeeze(-1)
        agirlik = torch.clamp(x / x_max, max=1.0) ** alfa
        return (agirlik * (tahmin - torch.log(x)) ** 2).mean()

    def vektorler(self):
        return (self.w.weight + self.w_tilde.weight).detach().cpu()

satir_i, sutun_j = X_es.nonzero(as_tuple=True)
degerler = X_es[satir_i, sutun_j]
print("Sıfır olmayan eş-oluşum hücresi:", len(degerler))

glove = GloVe(V, D).to(cihaz)
opt = torch.optim.Adagrad(glove.parameters(), lr=0.05)
glove_kayip = []
for epok in range(100):
    perm = torch.randperm(len(degerler))
    toplam = 0.0
    for b in range(0, len(perm), 1024):
        p = perm[b:b + 1024]
        kayip = glove(satir_i[p].to(cihaz), sutun_j[p].to(cihaz), degerler[p].to(cihaz))
        opt.zero_grad(); kayip.backward(); opt.step()
        toplam += kayip.item() * len(p)
    glove_kayip.append(toplam / len(perm))
    if (epok + 1) % 20 == 0:
        print(f"  epok {epok + 1}: kayıp = {glove_kayip[-1]:.4f}")

In [ ]:
glove_vektorleri = glove.vektorler()
print("GloVe en yakın komşular:")
display(komsu_tablosu(glove_vektorleri))

fig, eksenler = plt.subplots(1, 2, figsize=(15, 5.5))
pca_ciz(glove_vektorleri, "GloVe (PCA)", eksenler[0])
eksenler[1].plot(glove_kayip)
eksenler[1].set_title("GloVe eğitim kaybı"); eksenler[1].set_xlabel("epok"); eksenler[1].set_yscale("log")
plt.tight_layout(); plt.show()

---
## 4) FastText (Bojanowski vd., 2017)

Word2Vec ve GloVe her kelimeyi **bölünmez bir birim** olarak görür. Korpusta `kedi` varsa ama `kediler` yoksa, `kediler` için vektör üretemezler.

FastText her kelimeyi **karakter n-gramlarının toplamı** olarak temsil eder:

$$\mathbf{v}_{\text{kedi}} = \mathbf{z}_{\texttt{<ke}} + \mathbf{z}_{\texttt{ked}} + \mathbf{z}_{\texttt{edi}} + \mathbf{z}_{\texttt{di>}} + \dots + \mathbf{z}_{\texttt{<kedi>}}$$

- `<` ve `>` kelime başı/sonu işaretleridir (`edi` iç n-gramı ile `di>` son eki ayırt edilir).
- Milyonlarca n-gram için ayrı satır tutmak yerine n-gramlar **hash** ile sabit sayıda kovaya (bucket) eşlenir.
- Eğitim, skip-gram + negative sampling ile aynıdır; sadece merkez vektörü n-gram toplamından gelir.

In [ ]:
def karakter_ngramlari(kelime, n_min=3, n_max=5):
    k = f"<{kelime}>"
    ngramlar = [k[i:i + n] for n in range(n_min, n_max + 1) for i in range(len(k) - n + 1)]
    return ngramlar + [k]          # kelimenin kendisi de bir "n-gram" olarak eklenir

print("kedi    →", karakter_ngramlari("kedi"))
print("kediler →", karakter_ngramlari("kediler"))
ortak = set(karakter_ngramlari("kedi")) & set(karakter_ngramlari("kediler"))
print("\nOrtak n-gramlar:", sorted(ortak))

In [ ]:
KOVA = 20_000

def ngram_idleri(kelime):
    # Python'un hash()'i oturumlar arası değişebildiği için sabit bir FNV-1a hash kullanıyoruz
    idler = []
    for g in karakter_ngramlari(kelime):
        h = 2166136261
        for bayt in g.encode("utf-8"):
            h = ((h ^ bayt) * 16777619) % (2 ** 32)
        idler.append(h % KOVA)
    return idler

class FastTextSG(nn.Module):
    def __init__(self, kova, V, d):
        super().__init__()
        self.ngram = nn.EmbeddingBag(kova, d, mode="sum")
        self.cikis = nn.Embedding(V, d)
        nn.init.uniform_(self.ngram.weight, -0.5 / d, 0.5 / d)
        nn.init.zeros_(self.cikis.weight)

    def kelime_vektoru(self, ngram_listesi):
        duz = torch.tensor([i for lst in ngram_listesi for i in lst], device=self.cikis.weight.device)
        ofset = torch.tensor([0] + [len(l) for l in ngram_listesi[:-1]], device=duz.device).cumsum(0)
        return self.ngram(duz, ofset)

    def forward(self, merkez_ngramlari, baglam, negatif):
        v = self.kelime_vektoru(merkez_ngramlari)
        poz = F.logsigmoid((v * self.cikis(baglam)).sum(-1))
        neg = F.logsigmoid(-torch.bmm(self.cikis(negatif), v.unsqueeze(-1)).squeeze(-1)).sum(-1)
        return -(poz + neg).mean()

kelime_ngram = {i: ngram_idleri(w) for i, w in enumerate(sozluk)}

fasttext = FastTextSG(KOVA, V, D).to(cihaz)
opt = torch.optim.Adam(fasttext.parameters(), lr=0.01)
for epok in range(5):
    perm = torch.randperm(len(sg_ciftleri))
    toplam = 0.0
    for b in range(0, len(perm), 512):
        p = perm[b:b + 512]
        merkez, baglam = sg_ciftleri[p, 0], sg_ciftleri[p, 1].to(cihaz)
        neg = torch.multinomial(negatif_dagilim, len(p) * 5, replacement=True).view(len(p), 5).to(cihaz)
        kayip = fasttext([kelime_ngram[m] for m in merkez.tolist()], baglam, neg)
        opt.zero_grad(); kayip.backward(); opt.step()
        toplam += kayip.item() * len(p)
    print(f"  epok {epok + 1}: kayıp = {toplam / len(perm):.4f}")

### FastText'in süper gücü: sözlük dışı (OOV) kelimeler

`kediler`, `elmalar`, `ankaralı`, `trenler` korpusta **hiç geçmiyor**. Word2Vec bunlar için `KeyError` verirdi; FastText ise n-gramlardan bir vektör inşa eder.

In [ ]:
with torch.no_grad():
    ft_vektorleri = fasttext.kelime_vektoru([kelime_ngram[i] for i in range(V)]).cpu()

def oov_komsulari(kelime, n=4):
    with torch.no_grad():
        v = fasttext.kelime_vektoru([ngram_idleri(kelime)]).cpu()[0]
    E = normalize(ft_vektorleri)
    benzerlik = E @ (v / v.norm())
    sirali = benzerlik.argsort(descending=True)[:n]
    return [(sozluk[i], round(benzerlik[i].item(), 2)) for i in sirali.tolist()]

for kelime in ["kediler", "köpekçik", "elmalar", "kirazlı", "ankaralı", "trenler", "uçaklar"]:
    print(f"{kelime:10s} (sözlükte: {'evet' if kelime in kelime2id else 'HAYIR'}) → {oov_komsulari(kelime)}")

In [ ]:
print("FastText en yakın komşular (sözlük içi):")
display(komsu_tablosu(ft_vektorleri))

fig, ax = plt.subplots(figsize=(8, 6))
pca_ciz(ft_vektorleri, "FastText (PCA)", ax)
plt.show()

---
## 5) Nicel karşılaştırma: kategori ayrışması

Her yöntem için "aynı kategorideki kelimelerin ortalama kosinüs benzerliği" ile "farklı kategorilerdekilerin ortalama benzerliği" arasındaki farkı ölçelim. Fark büyükse vektör uzayı anlamsal kategorileri iyi ayırmış demektir.

In [ ]:
def kategori_ayrismasi(vektorler):
    hedef = [w for ws in kategoriler.values() for w in ws]
    E = normalize(vektorler)[[kelime2id[w] for w in hedef]]
    S = (E @ E.T).numpy()
    etiket = np.array([kelime_kategorisi[w] for w in hedef])
    ayni = (etiket[:, None] == etiket[None, :]) & ~np.eye(len(hedef), dtype=bool)
    farkli = etiket[:, None] != etiket[None, :]
    return S[ayni].mean(), S[farkli].mean()

satirlar = []
for isim, vek in [("Skip-gram", sg_vektorleri), ("CBOW", cbow_vektorleri),
                  ("GloVe", glove_vektorleri), ("FastText", ft_vektorleri)]:
    ayni, farkli = kategori_ayrismasi(vek)
    satirlar.append({"Yöntem": isim, "aynı kategori benzerliği": ayni,
                     "farklı kategori benzerliği": farkli, "ayrışma (fark)": ayni - farkli})
pd.DataFrame(satirlar).round(3)

---
## Özet

| | TF-IDF | Word2Vec | GloVe | FastText |
|---|---|---|---|---|
| Vektör tipi | Seyrek, $\lvert V\rvert$ boyutlu | Yoğun, ~100–300 boyut | Yoğun | Yoğun |
| Öğrenme sinyali | Sayım + IDF | **Yerel** pencere tahmini | **Küresel** eş-oluşum istatistiği | Yerel pencere + **alt-kelime** |
| Kelime benzerliği | Yok | Var | Var | Var |
| Sözlük dışı kelime | Yok sayılır | ❌ | ❌ | ✅ n-gramlardan |
| Türkçe için | Ekler ayrı kelime sayılır | Her çekim ayrı vektör | Her çekim ayrı vektör | **Ekleri paylaşır** |

Bu statik temsillerin ortak sınırlılığı: her kelimenin **tek** bir vektörü var (`yüz` = sayı mı, surat mı, yüzmek mi?). Bağlama göre değişen temsiller için → `transformers-llm/` klasöründeki Transformer defterleri.